# Notebook 3: Improved Experiments and Honest Evaluation

This is the improved version of notebook 2. **What changed:**

1. **3 random repeats (seeds)** instead of 1. The accuracy drops are small, so a single random split is very noisy. Results are averaged over the seeds.
2. **Your earlier results are reused.** The files you already have count as seed 0, so only seeds 1 and 2 need to run. **Do not delete anything in your Drive folder.**
3. **Five methods are compared**, including simpler meta-models that overfit less.
4. **Clean data rows are left out of the scoring.** Their accuracy drop is always 0, which is too easy to predict.
5. **Better evidence:** for each method we count on how many datasets it beats the simple baseline, and we run a statistical test.
6. **A breakdown by problem type** shows where the idea works and where it does not.

**How to run:** upload this notebook to Colab, click **Runtime → Run all**, and allow Google Drive access. Seeds 1 and 2 take roughly twice as long as the run you already did. If Colab disconnects, run it again and it continues where it stopped.

To test quickly first, set `FAST_TEST = True` in the third code cell (3 small datasets), and change it back afterwards.

In [ ]:
!pip install -q openml

## Setup: where results are saved

This is the same Drive folder as before (`data_quality_paper`), so your earlier results are found automatically.

In [ ]:
import os

# Results are saved here. In Colab this is a folder in your Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/data_quality_paper'
except ImportError:
    OUTPUT_DIR = 'results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Saving results to:', OUTPUT_DIR)

In [ ]:
import time
import glob
import warnings
import numpy as np
import pandas as pd
import openml

warnings.filterwarnings('ignore')

FAST_TEST = False       # True = only 3 small datasets, to test that everything works
SEEDS = [0, 1, 2]       # 3 repeats with different random splits
MODELS = ['logreg', 'rf', 'xgb']

DATASET_IDS = [3, 15, 29, 31, 37, 38, 44, 50, 1049, 1067, 1480,
               11, 23, 46, 54, 307, 12, 14, 16, 22, 28]
if FAST_TEST:
    DATASET_IDS = [15, 37, 11]   # breast-w, diabetes, balance-scale


def load_dataset(did):
    ds = openml.datasets.get_dataset(did)
    X, y, _, _ = ds.get_data(target=ds.default_target_attribute)
    # turn text columns into numbers, keep missing values as NaN
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            codes = X[col].astype('category').cat.codes.astype(float)
            X[col] = codes.where(codes >= 0, np.nan)
    X = X.astype(float).reset_index(drop=True)
    y = pd.factorize(y)[0]
    return X, y, ds.name

## Part A: data problems and models

Unchanged from notebook 2. The problems are always applied in this order: wrong labels, imbalance, duplicates, missing values.

In [ ]:
def inject_missing(X, level, seed=0):
    rng = np.random.default_rng(seed)
    mask = rng.random(X.shape) < level
    return X.mask(mask)


def inject_label_noise(y, level, seed=0):
    rng = np.random.default_rng(seed)
    y = y.copy()
    classes = np.unique(y)
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = rng.choice(classes[classes != y[i]])
    return y


def inject_duplicates(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), int(level * len(X)), replace=True)
    X2 = pd.concat([X, X.iloc[idx]], ignore_index=True)
    y2 = np.concatenate([y, y[idx]])
    return X2, y2


def inject_imbalance(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    biggest = classes[np.argmax(counts)]
    keep = []
    for c in classes:
        idx = np.where(y == c)[0]
        if c != biggest:
            n_keep = max(2, int(round(len(idx) * (1 - level))))
            idx = rng.choice(idx, n_keep, replace=False)
        keep.extend(idx)
    keep = np.sort(np.array(keep))
    return X.iloc[keep].reset_index(drop=True), y[keep]


def apply_defect(X_tr, y_tr, defect, level, seed=0):
    if defect == 'missing':
        return inject_missing(X_tr, level, seed), y_tr
    if defect == 'label_noise':
        return X_tr, inject_label_noise(y_tr, level, seed)
    if defect == 'duplicates':
        return inject_duplicates(X_tr, y_tr, level, seed)
    if defect == 'imbalance':
        return inject_imbalance(X_tr, y_tr, level, seed)
    raise ValueError('unknown defect: ' + defect)


DEFECTS = ['missing', 'label_noise', 'duplicates', 'imbalance']
DEFECT_ORDER = ['label_noise', 'imbalance', 'duplicates', 'missing']


def apply_defects(X, y, setting, seed=0):
    # setting is a dict such as {'missing': 0.2, 'label_noise': 0.1}
    for d in DEFECT_ORDER:
        if d in setting:
            X, y = apply_defect(X, y, d, setting[d], seed)
    return X, y


LEVELS = [0.05, 0.1, 0.2, 0.4]
COMBO_LEVELS = [0.1, 0.2]


def build_settings():
    settings = [{}]                       # the clean data (no problems)
    for d in DEFECTS:                     # one problem at a time
        for lv in LEVELS:
            settings.append({d: lv})
    for lv in COMBO_LEVELS:               # all four problems together
        settings.append({d: lv for d in DEFECTS})
    return settings


SETTINGS = build_settings()
print('Number of settings per dataset:', len(SETTINGS))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier


def make_model(name):
    imputer = SimpleImputer(strategy='median')
    if name == 'logreg':
        return make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    if name == 'rf':
        return make_pipeline(imputer, RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0))
    if name == 'xgb':
        return make_pipeline(imputer, XGBClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbosity=0))
    raise ValueError('unknown model: ' + name)

## Part B: meta-features

Unchanged from notebook 2. They are measured on the damaged training data, before any model is trained.

In [ ]:
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

META_COLS = ['n_rows', 'n_features', 'n_classes', 'rows_per_feature',
             'missing_rate', 'duplicate_rate', 'imbalance_ratio', 'outlier_rate',
             'mean_skew', 'mean_kurtosis', 'mean_abs_corr',
             'class_entropy', 'mean_mutual_info',
             'lm_stump', 'lm_tree3', 'lm_nb']


def compute_meta_features(X, y, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape
    classes, counts = np.unique(y, return_counts=True)
    prop = counts / counts.sum()

    mf = {
        'n_rows': n,
        'n_features': p,
        'n_classes': len(classes),
        'rows_per_feature': n / p,
        'missing_rate': float(X.isna().to_numpy().mean()),
        'duplicate_rate': float(X.assign(_y=y).duplicated().mean()),
        'imbalance_ratio': float(counts.min() / counts.max()),
        'class_entropy': float(entropy(prop) / np.log(len(classes))) if len(classes) > 1 else 0.0,
    }

    # outliers: share of cells more than 3 standard deviations from the column mean
    A = X.to_numpy(dtype=float)
    mu = np.nanmean(A, axis=0)
    sd = np.nanstd(A, axis=0)
    sd[sd == 0] = np.nan
    z = np.abs((A - mu) / sd)
    valid = ~np.isnan(z)
    mf['outlier_rate'] = float((z[valid] > 3).mean()) if valid.any() else 0.0

    mf['mean_skew'] = float(X.skew().mean())
    mf['mean_kurtosis'] = float(X.kurt().mean())

    if p > 1:
        C = X.corr().abs().to_numpy()
        mf['mean_abs_corr'] = float(np.nanmean(C[np.triu_indices(p, k=1)]))
    else:
        mf['mean_abs_corr'] = 0.0

    # small sample for the model-based features
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    Xs = X.iloc[idx]
    ys = y[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    cols = rng.choice(p, min(p, 20), replace=False)

    try:
        mf['mean_mutual_info'] = float(mutual_info_classif(Xi[:, cols], ys, random_state=seed).mean())
    except Exception:
        mf['mean_mutual_info'] = np.nan

    def landmark(model):
        try:
            return float(cross_val_score(model, Xi, ys, cv=3).mean())
        except Exception:
            return np.nan

    mf['lm_stump'] = landmark(DecisionTreeClassifier(max_depth=1, random_state=0))
    mf['lm_tree3'] = landmark(DecisionTreeClassifier(max_depth=3, random_state=0))
    mf['lm_nb'] = landmark(GaussianNB())
    return mf

## Part A (continued): run the experiments

One file is saved per dataset and seed. Seed 0 results from your earlier run are copied and reused, so only the missing work is done.

In [ ]:
def run_dataset(did, settings, seeds=(0,), models=MODELS):
    X, y, name = load_dataset(did)
    rows = []
    for seed in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=seed)
        X_tr = X_tr.reset_index(drop=True)

        clean_acc = {}
        for m in models:
            clean_acc[m] = accuracy_score(y_te, make_model(m).fit(X_tr, y_tr).predict(X_te))

        for sid, setting in enumerate(settings):
            Xd, yd = apply_defects(X_tr, y_tr, setting, seed)
            mf = compute_meta_features(Xd, yd, seed)
            for m in models:
                if setting:
                    acc = accuracy_score(y_te, make_model(m).fit(Xd, yd).predict(X_te))
                else:
                    acc = clean_acc[m]
                row = {'dataset': name, 'dataset_id': did, 'seed': seed,
                       'setting_id': sid, 'model': m}
                for d in DEFECTS:
                    row['inj_' + d] = setting.get(d, 0.0)
                row.update(mf)
                row['clean_acc'] = clean_acc[m]
                row['defect_acc'] = acc
                row['degradation'] = clean_acc[m] - acc
                rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
import shutil

for did in DATASET_IDS:
    legacy = os.path.join(OUTPUT_DIR, 'results_' + str(did) + '.csv')
    for seed in SEEDS:
        path = os.path.join(OUTPUT_DIR, 'results_' + str(did) + '_seed' + str(seed) + '.csv')
        if os.path.exists(path):
            continue
        # reuse the seed-0 results made by notebook 2
        if seed == 0 and os.path.exists(legacy):
            if pd.read_csv(legacy)['seed'].unique().tolist() == [0]:
                shutil.copy(legacy, path)
                print('reused earlier results for dataset', did, '(seed 0)')
                continue
        t0 = time.time()
        df = run_dataset(did, SETTINGS, seeds=[seed])
        df.to_csv(path, index=False)
        name = df['dataset'].iloc[0]
        minutes = (time.time() - t0) / 60
        print(f'done {did} ({name}), seed {seed}, in {minutes:.1f} min')

## Part C: prepare the results

- Results of the different seeds are **averaged**, which reduces the noise.
- Rows for the clean data (no problems added) are **removed**, because their accuracy drop is always exactly 0.
- Each row is labelled with its problem type (`missing`, `label_noise`, `duplicates`, `imbalance`, or `combined`).

In [ ]:
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'results_*_seed*.csv')))
all_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
all_df = all_df[all_df['dataset_id'].isin(DATASET_IDS) & all_df['seed'].isin(SEEDS)]
print('Datasets:', all_df['dataset_id'].nunique(), '| Seeds:', sorted(all_df['seed'].unique().tolist()))

agg_cols = META_COLS + ['clean_acc', 'defect_acc', 'degradation'] + ['inj_' + d for d in DEFECTS]
data = (all_df.groupby(['dataset', 'dataset_id', 'setting_id', 'model'], as_index=False)[agg_cols].mean())

# leave out the clean data (setting 0)
data = data[data['setting_id'] != 0].reset_index(drop=True)


def defect_type(row):
    active = [d for d in DEFECTS if row['inj_' + d] > 0]
    if len(active) == 1:
        return active[0]
    return 'combined'


data['defect_type'] = data.apply(defect_type, axis=1)
print('Rows used for scoring:', len(data))
data.head()

## Part C: compare five methods with leave-one-dataset-out

Leave-one-dataset-out means: hide one dataset, train on all the others, predict the hidden one, and repeat for every dataset. This shows whether the method works on **new data**.

| Method | What it is |
|---|---|
| `average drop` | **Baseline.** Always predicts the average accuracy drop seen in training |
| `defect-rate rule` | **Baseline.** Straight-line model using only `missing_rate`, `duplicate_rate`, `imbalance_ratio` |
| `ridge (small set)` | Meta-model: straight-line model with regularization, using 7 meta-features |
| `forest (small set)` | Meta-model: small, restricted random forest, using the same 7 meta-features |
| `forest (all features)` | Meta-model: random forest using all 16 meta-features |

The "small set" is the three defect rates plus `class_entropy` and the three landmarking features.

**How to read the summary table**
- **MAE:** average size of the mistake (lower is better).
- **rank_corr:** whether the damage is put in the right order (closer to 1 is better). Not shown for `average drop`, because it always predicts nearly the same number.
- **better_on:** on how many datasets the method has a smaller error than `average drop`. For example `15/21` means 15 of 21 datasets.
- **p_value:** from a paired Wilcoxon test against `average drop`. A value below 0.05 means the difference is unlikely to be just chance. You compare several methods, so treat a p-value only slightly below 0.05 with care.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr, wilcoxon

FEATURE_SETS = {
    'rule': ['missing_rate', 'duplicate_rate', 'imbalance_ratio'],
    'small': ['missing_rate', 'duplicate_rate', 'imbalance_ratio', 'class_entropy',
              'lm_stump', 'lm_tree3', 'lm_nb'],
    'all': META_COLS,
}

METHODS = {
    'average drop': None,
    'defect-rate rule': ('linear', 'rule'),
    'ridge (small set)': ('ridge', 'small'),
    'forest (small set)': ('forest_small', 'small'),
    'forest (all features)': ('forest', 'all'),
}
REFERENCE = 'average drop'


def make_meta(kind):
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    if kind == 'linear':
        return make_pipeline(imputer, LinearRegression())
    if kind == 'ridge':
        return make_pipeline(imputer, StandardScaler(), Ridge(alpha=1.0))
    if kind == 'forest_small':
        return make_pipeline(imputer, RandomForestRegressor(
            n_estimators=200, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=0))
    if kind == 'forest':
        return make_pipeline(imputer, RandomForestRegressor(
            n_estimators=200, n_jobs=-1, random_state=0))
    raise ValueError(kind)


def evaluate(d):
    y = d['degradation'].to_numpy()
    groups = d['dataset_id'].to_numpy()
    preds = {name: np.zeros(len(d)) for name in METHODS}
    for tr, te in LeaveOneGroupOut().split(d, y, groups):
        for name, spec in METHODS.items():
            if spec is None:
                preds[name][te] = y[tr].mean()
            else:
                kind, fs = spec
                cols = FEATURE_SETS[fs]
                preds[name][te] = make_meta(kind).fit(d.iloc[tr][cols], y[tr]).predict(d.iloc[te][cols])
    return preds


summary = []
store = {}
for m in MODELS:
    d = data[data['model'] == m].reset_index(drop=True)
    preds = evaluate(d)
    store[m] = (d, preds)

    err_by_ds = {}
    for name, pr in preds.items():
        err_by_ds[name] = pd.Series(np.abs(d['degradation'].to_numpy() - pr)).groupby(d['dataset_id']).mean()

    for name, pr in preds.items():
        row = {'model': m, 'method': name,
               'MAE': mean_absolute_error(d['degradation'], pr),
               'rank_corr': np.nan if name == REFERENCE else spearmanr(d['degradation'], pr)[0]}
        if name != REFERENCE:
            diff = err_by_ds[name] - err_by_ds[REFERENCE]
            row['better_on'] = str(int((diff < 0).sum())) + '/' + str(len(diff))
            try:
                row['p_value'] = wilcoxon(err_by_ds[name], err_by_ds[REFERENCE]).pvalue
            except Exception:
                row['p_value'] = np.nan
        summary.append(row)

summary_df = pd.DataFrame(summary).round(3)
summary_df

### Where does it work? Breakdown by problem type

Each number is the average mistake (MAE) for that type of problem. `real_mean_drop` is the average real accuracy drop, so you can see how big the mistakes are compared with the thing being predicted. If a method's mistake is much smaller than `real_mean_drop` and smaller than `average drop`, the method works for that problem type.

In [ ]:
rows = []
for m in MODELS:
    d, preds = store[m]
    t = d.groupby('defect_type')['degradation'].mean().rename('real_mean_drop').to_frame()
    for name, pr in preds.items():
        t[name] = pd.Series(np.abs(d['degradation'].to_numpy() - pr)).groupby(d['defect_type']).mean()
    t.insert(0, 'model', m)
    rows.append(t.reset_index())

breakdown = pd.concat(rows, ignore_index=True).round(3)
breakdown

### Predicted vs real accuracy drop

Each dot is one dataset with one problem setting. Dots close to the dashed line mean good predictions. Change `BEST_METHOD` to see another method.

In [ ]:
import matplotlib.pyplot as plt

BEST_METHOD = 'forest (small set)'

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4))
for ax, m in zip(np.atleast_1d(axes), MODELS):
    d, preds = store[m]
    real = d['degradation'].to_numpy()
    pred = preds[BEST_METHOD]
    lo, hi = min(real.min(), pred.min()), max(real.max(), pred.max())
    ax.scatter(real, pred, s=12, alpha=0.6)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
    ax.set_title(m + ' - ' + BEST_METHOD)
    ax.set_xlabel('real accuracy drop')
    ax.set_ylabel('predicted accuracy drop')
plt.tight_layout()
plt.show()

### Which meta-features matter most?

A longer bar means the random forest relies on that number more. This answers research question 2.

In [ ]:
imp = pd.DataFrame(index=META_COLS)
for m in MODELS:
    d = data[data['model'] == m]
    pipe = make_meta('forest').fit(d[META_COLS], d['degradation'])
    imp[m] = pipe[-1].feature_importances_

imp['average'] = imp.mean(axis=1)
imp['average'].sort_values().plot.barh(figsize=(6, 5), title='Meta-feature importance (average)')
plt.tight_layout()
plt.show()

In [ ]:
data.to_csv(os.path.join(OUTPUT_DIR, 'final_data_averaged.csv'), index=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'final_summary.csv'), index=False)
breakdown.to_csv(os.path.join(OUTPUT_DIR, 'final_breakdown.csv'), index=False)
imp.to_csv(os.path.join(OUTPUT_DIR, 'final_feature_importance.csv'))
print('Saved final_*.csv files in', OUTPUT_DIR)

## How to read the result

1. **Does any meta-model beat both baselines?** Look at MAE, `better_on` and `p_value` together. A convincing result needs a lower MAE and a clearly small p-value, not just a small difference in one number.
2. **If yes:** use the summary table, the breakdown table, the scatter plot and the importance chart in your paper.
3. **If no:** that is still a valid and honest finding. You can report that simple defect-rate rules are hard to beat, and use the breakdown table to say which problem types are predictable and which are not.
4. Send me the summary table and the breakdown table, and I'll help you decide how to write it up.